# 01 — Data Quality and EDA

Quality checks, leakage analysis and baseline dataset creation.

**Project:** Marketing Analytics Causal & LTV Lab  
**Phase:** Phase 1 — Customer Analytics, Retention, Churn and LTV Baseline

> This notebook is designed as a hands-on learning notebook. Run each section, inspect the output, and discuss the interpretation before moving to the next step.


## 1. Notebook objective

This notebook builds the foundation for Phase 1.

We will:

- Load the Digital Wallet LTV dataset
- Check schema, missing values, duplicates and outliers
- Separate numeric and categorical features
- Inspect the target variable `LTV`
- Identify possible leakage features
- Save a clean baseline dataset

Important: this is a customer-level snapshot dataset. It is useful for LTV, churn proxy and segmentation, but it is not enough for true causal inference or MMM.


In [ ]:
# Core imports
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix
)
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_RAW = Path("../data/raw/digital_wallet_ltv_dataset.csv")
DATA_PROCESSED = Path("../data/processed")
REPORTS = Path("../reports")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_RAW)
df.head()


## 2. Basic schema and quality checks


In [ ]:
print("Shape:", df.shape)
display(df.head())
display(df.info())
display(df.describe(include="all").T)


In [ ]:
# Missing values
missing = (
    df.isna().mean()
    .sort_values(ascending=False)
    .rename("missing_rate")
    .to_frame()
)
display(missing)

# Duplicate customer IDs
if "Customer_ID" in df.columns:
    print("Duplicate Customer_ID rows:", df.duplicated("Customer_ID").sum())
else:
    print("Customer_ID column not found.")


## 3. Feature type detection

We explicitly separate ID, target, numeric and categorical features. This is important for clean ML pipelines later.


In [ ]:
ID_COL = "Customer_ID"
TARGET = "LTV"

numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_features = [c for c in numeric_cols if c not in [TARGET]]
categorical_features = [c for c in categorical_cols if c != ID_COL]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## 4. Target distribution


In [ ]:
plt.figure(figsize=(8, 4))
df[TARGET].hist(bins=40)
plt.title("LTV Distribution")
plt.xlabel("LTV")
plt.ylabel("Customer count")
plt.show()

print(df[TARGET].describe())


## 5. Outlier inspection

Marketing/customer datasets often contain heavy-tailed monetary behavior. Outliers are not automatically errors. They may be high-value customers.


In [ ]:
for col in ["Total_Spent", "Avg_Transaction_Value", "Max_Transaction_Value", "Total_Transactions", "LTV"]:
    if col in df.columns:
        q01, q99 = df[col].quantile([0.01, 0.99])
        print(f"{col}: p01={q01:.2f}, p99={q99:.2f}, max={df[col].max():.2f}")


## 6. Correlation and leakage risk

For LTV modeling, some variables may be too close to the target definition.

Potential leakage candidates:

- `Total_Spent`
- `Loyalty_Points_Earned`
- `Cashback_Received`
- `Total_Transactions`

They may still be valid depending on the prediction time window. In later notebooks we will run two versions:

1. **Business snapshot model** using all available behavioral data
2. **Leakage-aware early prediction model** excluding suspicious post-outcome variables


In [ ]:
corr = df.select_dtypes(include=["number"]).corr(numeric_only=True)

if TARGET in corr.columns:
    target_corr = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
    display(target_corr.to_frame("corr_with_LTV"))

    plt.figure(figsize=(8, 5))
    target_corr.head(12).sort_values().plot(kind="barh")
    plt.title("Top correlations with LTV")
    plt.xlabel("Correlation")
    plt.show()


## 7. Basic cleaning and save processed baseline


In [ ]:
df_clean = df.copy()

# Drop duplicate Customer_ID rows if any
if ID_COL in df_clean.columns:
    df_clean = df_clean.drop_duplicates(subset=[ID_COL])

# Strip whitespace from categorical columns
for col in categorical_features:
    df_clean[col] = df_clean[col].astype(str).str.strip()

output_path = DATA_PROCESSED / "wallet_clean.csv"
df_clean.to_csv(output_path, index=False)

print(f"Saved clean dataset to: {output_path}")
print("Clean shape:", df_clean.shape)


## Discussion prompts

Use these in our hands-on discussion:

1. Which features look suspiciously close to `LTV`?
2. Is this dataset suitable for true cohort analysis?
3. Which columns look like early behavior and which look like lifetime accumulated behavior?
4. What business question can we answer confidently from this dataset?
